# 04 Processing: raw collection → analysis-ready tables (project 1.3)

Turns what the collectors saved in `data_raw/` into tidy Parquet tables in `data_processed/` that R (or Python) can analyse. Run top to bottom whenever you want to refresh; finished days are cached, so reruns are fast.

What it builds
1. `link_registry.parquet`: one row per link (road name, category, coordinates, first/last seen) from the hourly raw snapshots.
2. `panel/sb_<date>.parquet`: speed bands for the selected road categories, one row per link × LTA update.
3. Quality report: snapshots per day, gaps, coverage, missing bands.
4. `incidents.parquet`: each incident with first/last time it was live and its reported time.
5. `analysis_panel/<date>.parquet`: the panel joined with nearest-station rainfall and calendar flags.

What it deliberately does **not** do: compute "normal" speeds, labels or anything learned from the data. Those must use the training weeks only (Brief §4: preprocessing on training data only), so they belong in the modelling step.

Needs: `pip install polars numpy matplotlib`. Run `get_context_data.py` first for rainfall and calendar. Python kernel, project folder as working directory.

AI disclosure: drafted with Claude (Anthropic), 14 Oct 2026; log in `AI_USE.md`.

## 1. Settings and helpers

In [ ]:
import gzip, json, re
from datetime import date, datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

PROJ = Path.cwd()
RAW, OUT = PROJ / "data_raw", PROJ / "data_processed"
CATEGORIES = [1]                 # 1 = expressways (primary analysis); add 2 for major arterials
FIRST_DAY = date(2026, 10, 13)   # collection window start
REBUILD = False                  # True = ignore the cache and rebuild every day

for d in ["registry_parts", "panel", "analysis_panel"]:
    (OUT / d).mkdir(parents=True, exist_ok=True)

TODAY = datetime.now().date()


def parse_sgt(col: pl.Expr) -> pl.Expr:
    """Timestamps like '2026-10-15 08:00:00', '2026-10-15T08:00:00+08:00' or '...+0800' -> naive Singapore time."""
    s = col.str.replace(r"(\+08:?00)$", "").str.replace("T", " ")
    return pl.coalesce(s.str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False),
                       s.str.to_datetime("%Y-%m-%d %H:%M:%S%.f", strict=False),
                       s.str.to_datetime("%Y-%m-%d %H:%M", strict=False))


def read_raw_pages(path: Path) -> list[dict]:
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def day_dirs(folder: Path) -> list[Path]:
    return sorted(p for p in folder.glob("????-??-??") if p.is_dir())


print("Raw speed-band days:", [p.name for p in day_dirs(RAW / "speedbands")])

## 2. Link registry
One row per link from the hourly full raw snapshots. Each raw file is summarised once and cached in `registry_parts/`.

In [ ]:
raw_files = sorted((RAW / "speedbands_raw").rglob("sbraw_*.jsonl.gz"))
cols = ["LinkID", "RoadName", "RoadCategory", "StartLon", "StartLat", "EndLon", "EndLat"]
new = 0
for f in raw_files:
    part = OUT / "registry_parts" / (f.name.replace(".jsonl.gz", ".parquet"))
    if part.exists() and not REBUILD:
        continue
    pages = read_raw_pages(f)
    recs = [{k: r.get(k) for k in cols} for p in pages for r in json.loads(p["body"])["value"]]
    (pl.DataFrame(recs, schema={c: pl.Utf8 for c in cols})
       .with_columns(pl.lit(pages[0]["retrieved_at"]).alias("retrieved_at"))
       .write_parquet(part))
    new += 1
parts = pl.scan_parquet(OUT / "registry_parts" / "*.parquet").with_columns(
    parse_sgt(pl.col("retrieved_at")).alias("seen"), pl.col("RoadCategory").cast(pl.Int64))
n_files = parts.select(pl.col("seen").n_unique()).collect().item()
registry = (parts.sort("seen")
            .group_by("LinkID")
            .agg(pl.col("RoadName").last(), pl.col("RoadCategory").last(),
                 *[pl.col(c).cast(pl.Float64).last() for c in ["StartLon", "StartLat", "EndLon", "EndLat"]],
                 pl.col("seen").min().alias("first_seen"), pl.col("seen").max().alias("last_seen"),
                 pl.len().alias("n_snapshots"),
                 pl.struct("RoadName", "StartLon", "StartLat", "EndLon", "EndLat").n_unique().alias("n_attribute_versions"))
            .with_columns((pl.col("n_snapshots") / n_files).alias("presence"))
            .collect())
registry.write_parquet(OUT / "link_registry.parquet")
print(f"{len(raw_files)} raw files ({new} new); {registry.height:,} links seen")
print(registry.group_by("RoadCategory").agg(pl.len().alias("links"),
      (pl.col("presence") >= 0.95).sum().alias("present_95pct"),
      (pl.col("n_attribute_versions") > 1).sum().alias("attributes_changed")).sort("RoadCategory"))

## 3. Speed-band panel for the selected categories
One file per day. Rows are deduplicated on (link, LTA update time). Band 0 means "no data" and becomes missing. Today is always rebuilt because it is still growing.

In [ ]:
selected = registry.filter(pl.col("RoadCategory").is_in(CATEGORIES)).select("LinkID")
ids = selected["LinkID"]
print(f"Selected links (categories {CATEGORIES}): {ids.len():,}")

for ddir in day_dirs(RAW / "speedbands"):
    day = date.fromisoformat(ddir.name)
    out = OUT / "panel" / f"sb_{day}.parquet"
    if out.exists() and not REBUILD and day < TODAY:
        continue
    frames = []
    for f in sorted(ddir.glob("sb_*.csv.gz")):
        with gzip.open(f, "rb") as fh:
            df = pl.read_csv(fh.read(), schema_overrides={"LinkID": pl.Utf8, "SpeedBand": pl.Int64,
                                                          "retrieved_at": pl.Utf8, "last_updated": pl.Utf8})
        frames.append(df.filter(pl.col("LinkID").is_in(ids)))
    if not frames:
        continue
    panel = (pl.concat(frames)
               .with_columns(parse_sgt(pl.col("retrieved_at")).alias("retrieved"),
                             parse_sgt(pl.col("last_updated")).alias("lta_time"))
               .with_columns(pl.coalesce("lta_time", "retrieved").alias("t"),
                             pl.when(pl.col("SpeedBand").is_between(1, 8)).then(pl.col("SpeedBand")).cast(pl.Int8).alias("band"))
               .sort("retrieved")
               .unique(subset=["LinkID", "t"], keep="first")
               .select("t", "LinkID", "band", "retrieved", pl.col("lta_time").is_null().alias("t_from_retrieval"))
               .sort("t", "LinkID"))
    panel.write_parquet(out)
    print(f"{day}: {panel['t'].n_unique()} updates, {panel.height:,} rows")

sample = pl.read_parquet(sorted((OUT / "panel").glob("*.parquet"))[-1]).head(3)
print(sample)

## 4. Quality report
Targets: about 288 updates per day, no gap over 10 minutes, link coverage near 100%, few missing bands.

In [ ]:
panel_all = pl.scan_parquet(OUT / "panel" / "*.parquet")
n_links = ids.len()
quality = (panel_all.group_by(pl.col("t").dt.date().alias("date"))
           .agg(pl.col("t").n_unique().alias("updates"),
                (pl.len() / pl.col("t").n_unique() / n_links).alias("link_coverage"),
                pl.col("band").is_null().mean().alias("share_missing_band"),
                pl.col("t_from_retrieval").mean().alias("share_time_from_retrieval"))
           .sort("date").collect())
times = panel_all.select(pl.col("t").unique().sort()).collect()["t"]
gaps = (times.diff().dt.total_minutes()).alias("gap_min")
gap_tbl = pl.DataFrame({"t": times, "gap_min": gaps}).filter(pl.col("gap_min") > 10)
print(quality)
print(f"Gaps over 10 minutes: {gap_tbl.height}")
print(gap_tbl.sort("gap_min", descending=True).head(10))

# Updates per hour, day x hour (12 = complete)
cov = (pl.DataFrame({"t": times}).group_by(pl.col("t").dt.date().alias("date"), pl.col("t").dt.hour().alias("hour"))
       .len().pivot(on="hour", index="date", values="len").sort("date").fill_null(0))
hours = [str(h) for h in range(24) if str(h) in cov.columns]
fig, ax = plt.subplots(figsize=(12, 0.35 * cov.height + 1.5))
im = ax.imshow(cov.select(hours).to_numpy(), aspect="auto", cmap="Greens", vmin=0, vmax=12)
ax.set_yticks(range(cov.height), [str(d) for d in cov["date"]]); ax.set_xticks(range(len(hours)), hours)
ax.set_xlabel("Hour (SGT)"); ax.set_title("LTA updates captured per hour (12 = complete)", loc="left")
fig.colorbar(im, ax=ax, fraction=0.02); fig.tight_layout()
fig.savefig(PROJ / "figures" / "quality_updates_per_hour.png", dpi=150, facecolor="white") if (PROJ / "figures").exists() else None
plt.show()

## 5. Incidents
Each saved incidents file is a change in the live list. An incident's `first_seen` is the first file it appears in; `last_seen_before` is the first later file where it is gone. The report time is parsed from the message, e.g. `(13/10)21:34 ...`.

In [ ]:
inc_files = sorted((RAW / "feeds" / "incidents").rglob("incidents_*.jsonl.gz"))
snapshots = []
for f in inc_files:
    pages = read_raw_pages(f)
    t = pages[0]["retrieved_at"]
    recs = [r for p in pages for r in json.loads(p["body"]).get("value", [])]
    snapshots.append((t, {(r.get("Type"), r.get("Message"), r.get("Latitude"), r.get("Longitude")) for r in recs}))
rows = []
for i, (t, live) in enumerate(snapshots):
    prev = snapshots[i - 1][1] if i else set()
    for key in live - prev:  # appears
        gone = next((snapshots[j][0] for j in range(i + 1, len(snapshots)) if key not in snapshots[j][1]), None)
        rows.append({"type": key[0], "message": key[1], "lat": key[2], "lon": key[3], "first_seen": t, "last_seen_before": gone})
incidents = pl.DataFrame(rows, schema={"type": pl.Utf8, "message": pl.Utf8, "lat": pl.Float64, "lon": pl.Float64,
                                       "first_seen": pl.Utf8, "last_seen_before": pl.Utf8})
if incidents.height:
    year = str(FIRST_DAY.year)
    incidents = (incidents
        .with_columns(parse_sgt(pl.col("first_seen")).alias("first_seen"), parse_sgt(pl.col("last_seen_before")).alias("last_seen_before"),
                      pl.col("message").str.extract(r"^\((\d{1,2}/\d{1,2})\)(\d{1,2}:\d{2})", 1).alias("_d"),
                      pl.col("message").str.extract(r"^\((\d{1,2}/\d{1,2})\)(\d{1,2}:\d{2})", 2).alias("_t"))
        .with_columns((pl.col("_d") + "/" + year + " " + pl.col("_t")).str.to_datetime("%d/%m/%Y %H:%M", strict=False).alias("reported"))
        .drop("_d", "_t"))
incidents.write_parquet(OUT / "incidents.parquet")
print(f"{len(inc_files)} incident files -> {incidents.height} incidents")
print(incidents.group_by("type").len().sort("len", descending=True) if incidents.height else "no incidents yet")

## 6. Rainfall and calendar joins → analysis panel
Each link gets its nearest rain station (straight-line distance from the link midpoint). `rain_mm` is the 5-minute total for the slot ending at that time. Feature engineering (e.g. rain over the last 30 minutes, lagged by 5 minutes for publication delay) happens in the modelling step.

In [ ]:
stations = pl.read_csv(OUT / "rain_stations.csv", schema_overrides={"station_id": pl.Utf8})
links_sel = registry.filter(pl.col("RoadCategory").is_in(CATEGORIES)).with_columns(
    ((pl.col("StartLon") + pl.col("EndLon")) / 2).alias("mid_lon"), ((pl.col("StartLat") + pl.col("EndLat")) / 2).alias("mid_lat"))

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

dist = haversine_km(links_sel["mid_lat"].to_numpy()[:, None], links_sel["mid_lon"].to_numpy()[:, None],
                    stations["lat"].to_numpy()[None, :], stations["lon"].to_numpy()[None, :])
nearest = links_sel.select("LinkID", "RoadName").with_columns(
    pl.Series("station_id", stations["station_id"].to_numpy()[dist.argmin(axis=1)]),
    pl.Series("station_km", dist.min(axis=1).round(2)))
nearest.write_parquet(OUT / "link_nearest_station.parquet")
print(f"Nearest station distance (km): median {nearest['station_km'].median():.2f}, max {nearest['station_km'].max():.2f}")

rain = (pl.read_csv(OUT / "rainfall_5min.csv.gz", schema_overrides={"station_id": pl.Utf8, "timestamp": pl.Utf8, "rain_mm": pl.Float64})
          .with_columns(parse_sgt(pl.col("timestamp")).alias("rain_t")).select("rain_t", "station_id", "rain_mm"))
cal = (pl.read_csv(OUT / "calendar_2026.csv", try_parse_dates=True)
         .select(pl.col("date").cast(pl.Date), "daytype", "is_public_holiday", "holiday_name", "school_period",
                 "olevel_venue_schools_ended", "f1_closure_day"))

for f in sorted((OUT / "panel").glob("sb_*.parquet")):
    day = date.fromisoformat(f.stem[3:])
    out = OUT / "analysis_panel" / f"{day}.parquet"
    if out.exists() and not REBUILD and day < TODAY - timedelta(days=1):   # rainfall for yesterday may arrive late
        continue
    ap = (pl.read_parquet(f)
            .join(nearest.select("LinkID", "station_id", "station_km"), on="LinkID", how="left")
            .with_columns(pl.col("t").dt.truncate("5m").alias("slot"), pl.col("t").dt.date().alias("date"))
            .join(rain, left_on=["slot", "station_id"], right_on=["rain_t", "station_id"], how="left")
            .join(cal, on="date", how="left")
            .with_columns(pl.col("t").dt.weekday().alias("weekday"),
                          (pl.col("t").dt.hour() * 4 + pl.col("t").dt.minute() // 15).cast(pl.Int16).alias("slot15")))
    ap.write_parquet(out)
    print(f"{day}: {ap.height:,} rows, rain matched {ap['rain_mm'].is_not_null().mean():.0%}")

final = pl.scan_parquet(OUT / "analysis_panel" / "*.parquet")
print(final.collect_schema())
print(final.select(pl.len().alias("rows"), pl.col("LinkID").n_unique().alias("links"),
                   pl.col("t").min().alias("from"), pl.col("t").max().alias("to")).collect())

## 7. Reading the output in R
```r
library(arrow); library(dplyr)
panel <- open_dataset("data_processed/analysis_panel") |> collect()
links <- read_parquet("data_processed/link_registry.parquet")
```
Columns: `t` (LTA update time, Singapore), `LinkID`, `band` (1–8, NA = no data), `station_id`, `station_km`, `slot`, `rain_mm`, calendar flags (`daytype`, `is_public_holiday`, `school_period`, ...), `weekday` (1 = Mon), `slot15` (15-minute slot of the day, 0–95).